# Atividade 03 - Temperatura e Criatividade

**Conceito:** Temperatura e um parametro que controla o quanto de "aleatoriedade" o modelo aceita
ao escolher a proxima palavra. Pense assim:
- **Temperatura 0** -> sempre escolhe a palavra mais provavel -> robo previsivel
- **Temperatura 1** -> balanceado entre previsibilidade e criatividade
- **Temperatura 2** -> aceita palavras improvaveis -> artista (as vezes genial, as vezes maluco)

> **Range de temperatura no Claude:** 0.0 a 2.0 (diferente do GPT que vai de 0 a 1)

**Objetivo desta atividade:**
- Observar como a temperatura afeta respostas factuais vs. criativas
- Medir a diversidade de respostas em funcao da temperatura
- Gerar um grafico temperatura x criatividade
- Descobrir a temperatura ideal para diferentes tarefas

---
> **Atencao:** esta atividade faz **muitas chamadas a API** (ate 55 chamadas no experimento).
> Cada chamada tem custo baixo, mas verifique seu saldo em https://console.anthropic.com

## Setup

In [ ]:
%pip install anthropic matplotlib -q

import anthropic
import os
import time
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from collections import Counter
from typing import List, Dict, Any

# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURACAO - Edite apenas esta secao
# ═══════════════════════════════════════════════════════════════════════════
os.environ["ANTHROPIC_API_KEY"] = "SUA_CHAVE_AQUI"  # <- Sua chave aqui
MODEL_NAME = "claude-sonnet-4-6"
# ═══════════════════════════════════════════════════════════════════════════

client = anthropic.Anthropic()

# Cache para evitar chamadas repetidas (util se precisar re-executar celulas)
_cache: Dict[str, str] = {}

def perguntar(prompt: str, temperatura: float = 1.0, max_tokens: int = 120, 
              usar_cache: bool = False) -> str:
    """
    Chama o Claude com temperatura configuravel.
    
    Args:
        prompt: Texto da pergunta
        temperatura: 0.0 (deterministico) a 2.0 (muito criativo)
        max_tokens: Limite de tokens na resposta
        usar_cache: Se True, reutiliza respostas anteriores (para temp=0)
    """
    cache_key = f"{prompt}:{temperatura}" if usar_cache else None
    if cache_key and cache_key in _cache:
        return _cache[cache_key]
    
    try:
        r = client.messages.create(
            model=MODEL_NAME,
            max_tokens=max_tokens,
            temperature=temperatura,
            messages=[{"role": "user", "content": prompt}]
        )
        resultado = r.content[0].text.strip()
        if cache_key:
            _cache[cache_key] = resultado
        return resultado
    except anthropic.RateLimitError:
        print("[AVISO] Rate limit! Aguardando 30s...")
        time.sleep(30)
        return perguntar(prompt, temperatura, max_tokens, usar_cache)
    except Exception as e:
        return f"[ERRO] {e}"

def perguntar_n_vezes(prompt: str, temperatura: float, n: int = 5) -> List[str]:
    """Chama o Claude N vezes com a mesma temperatura e retorna lista de respostas."""
    respostas = []
    for i in range(n):
        respostas.append(perguntar(prompt, temperatura))
        time.sleep(0.3)  # evita rate limit
    return respostas

print("[OK] Setup concluido!")

---
## Parte 1 - Temperatura em perguntas factuais

Para perguntas com resposta certa, a temperatura deveria mudar alguma coisa?

In [ ]:
pergunta_factual = "Qual é a capital do Brasil? Responda em uma única palavra."

print("=" * 65)
print("PERGUNTA FACTUAL: 'Qual é a capital do Brasil?'")
print("=" * 65)

for temp in [0.0, 0.5, 1.0, 1.5]:
    respostas = perguntar_n_vezes(pergunta_factual, temp, n=5)
    unicos    = set(r.lower().strip('.!?') for r in respostas)
    print(f"\nTemperatura {temp} → {len(unicos)}/5 respostas únicas")
    for r in respostas:
        print(f"  → {r}")

### O que voce observou?
Mesmo com temperatura alta, o modelo acerta? Por que?
*(Dica: pense na distribuicao de probabilidades - "Brasilia" tem probabilidade tao alta que mesmo com ruido ela vence)*

---
## Parte 2 - Temperatura em tarefas criativas

Agora vamos ver onde a temperatura realmente faz diferenca.

In [ ]:
prompt_criativo = "Complete em uma frase criativa: 'A inteligência artificial é como...'"

print("=" * 65)
print("TAREFA CRIATIVA")
print("=" * 65)

for temp in [0.0, 0.5, 1.0, 1.5]:
    respostas = perguntar_n_vezes(prompt_criativo, temp, n=3)
    print(f"\nTemperatura {temp}:")
    for r in respostas:
        print(f"  → {r}")

In [ ]:
# ─── Comparando um exemplo concreto de escrita criativa ───
prompt_historia = "Escreva o início de uma história de ficção científica em 2 frases."

print("Temperatura 0.0 (previsível):")
print(perguntar(prompt_historia, temperatura=0.0))

print("\nTemperatura 1.0 (balanceado):")
print(perguntar(prompt_historia, temperatura=1.0))

print("\nTemperatura 1.8 (caótico):")
print(perguntar(prompt_historia, temperatura=1.8))

---
## Desafio - Grafico Temperatura x Diversidade

Vamos **medir** a criatividade de forma sistematica.
A metrica que usaremos: **diversidade de vocabulario** = palavras unicas / total de palavras.

**Complete as partes marcadas com `# TODO`**

In [ ]:
def calcular_diversidade(textos: List[str]) -> float:
    """
    Calcula a diversidade de vocabulário de uma lista de textos.
    Diversidade = palavras únicas / total de palavras.
    Quanto mais alto, mais diverso (mais criativo).
    
    Args:
        textos: Lista de strings para analisar
        
    Returns:
        float entre 0.0 (repetitivo) e 1.0 (muito diverso)
    """
    # TODO 1: junte todos os textos em uma string, converta para minúsculas
    # e divida em palavras individuais
    # Dica: " ".join(textos).lower().split()
    todas_palavras = None  # ← substitua

    # TODO 2: trate o caso de lista vazia (retorne 0.0)
    if not todas_palavras:
        return 0.0

    # TODO 3: calcule e retorne palavras_unicas / total_palavras
    # Dica: use set() para obter palavras únicas
    return None  # ← substitua


# ─── Teste da função ───
teste_baixa = ["gato gato gato gato", "gato gato"]
teste_alta  = ["gato cachorro peixe hamster", "leão tigre urso lobo"]

print(f"Diversidade baixa esperada (~0.2): {calcular_diversidade(teste_baixa) or 0:.2f}")
print(f"Diversidade alta esperada (~1.0):  {calcular_diversidade(teste_alta) or 0:.2f}")

In [ ]:
# --- Experimento principal ---
# Testamos temperaturas de 0.0 a 2.0 em passos de 0.2
# Para cada temperatura, geramos 5 titulos de redacao

prompt_titulo = "Gere um titulo criativo para uma redacao do ENEM sobre tecnologia e sociedade. Apenas o titulo, sem explicacoes."

temperaturas  = [round(t * 0.2, 1) for t in range(11)]  # [0.0, 0.2, ..., 2.0]
diversidades  = []
todos_titulos = {}  # guarda os titulos para analise

# Tenta carregar resultados anteriores (cache)
CACHE_FILE = "experimento_temperatura_cache.json"
try:
    with open(CACHE_FILE, 'r', encoding='utf-8') as f:
        cached = json.load(f)
        todos_titulos = {float(k): v for k, v in cached.get('titulos', {}).items()}
        diversidades = cached.get('diversidades', [])
        print("[CACHE] Carregado do cache!")
except FileNotFoundError:
    todos_titulos = {}
    diversidades = []

# Se nao tem cache, roda o experimento
if not todos_titulos:
    print("[INFO] Rodando experimento... (pode demorar ~2 minutos)")
    print(f"{'Temperatura':>12} {'Diversidade':>12}  Exemplo de titulo")
    print("-" * 75)

    for temp in temperaturas:
        titulos = perguntar_n_vezes(prompt_titulo, temp, n=5)
        div = calcular_diversidade(titulos) or 0
        diversidades.append(div)
        todos_titulos[temp] = titulos
        print(f"{temp:>12} {div:>12.3f}  {titulos[0][:50]}")

    # Salva cache
    with open(CACHE_FILE, 'w', encoding='utf-8') as f:
        json.dump({
            'titulos': {str(k): v for k, v in todos_titulos.items()},
            'diversidades': diversidades
        }, f, ensure_ascii=False, indent=2)
    print(f"\n[SALVO] Resultados salvos em '{CACHE_FILE}' (delete para re-executar)")
else:
    print(f"{'Temperatura':>12} {'Diversidade':>12}")
    print("-" * 30)
    for temp, div in zip(temperaturas, diversidades):
        print(f"{temp:>12} {div:>12.3f}")

In [ ]:
# ─── Plotando o gráfico ───

# TODO 4: complete o gráfico adicionando:
# - Uma linha vertical pontilhada indicando "zona ideal" (entre 0.7 e 1.3)
# - Anotações de texto nos pontos extremos (temperatura 0.0 e 2.0)
# - Uma cor de fundo diferente para a zona ideal

fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(temperaturas, diversidades, marker='o', linewidth=2.5,
        color='royalblue', markersize=8, label='Diversidade de vocabulário')

# TODO: adicione zona ideal com ax.axvspan(0.7, 1.3, alpha=0.1, color='green', label='Zona ideal')
# TODO: adicione anotação no ponto 0.0: ax.annotate('Previsível', xy=(0.0, diversidades[0]), ...)
# TODO: adicione anotação no ponto 2.0: ax.annotate('Caótico',    xy=(2.0, diversidades[-1]), ...)

ax.set_xlabel("Temperatura", fontsize=12)
ax.set_ylabel("Diversidade de vocabulário", fontsize=12)
ax.set_title("Temperatura × Criatividade no Claude", fontsize=14, fontweight='bold')
ax.set_xticks(temperaturas)
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.savefig("temperatura_criatividade.png", dpi=150)
plt.show()
print("Gráfico salvo como 'temperatura_criatividade.png'")

In [ ]:
# ─── Análise dos títulos gerados ───
# Veja os títulos em diferentes temperaturas para entender qualitativamente

print("TÍTULOS GERADOS POR TEMPERATURA\n")
for temp in [0.0, 0.7, 1.0, 1.5, 2.0]:
    print(f"── Temperatura {temp} ──")
    for t in todos_titulos[temp]:
        print(f"  • {t}")
    print()

---
## Desafio Extra - Temperatura ideal por tarefa

Teste 3 tarefas diferentes e descubra empiricamente qual temperatura e melhor para cada uma.

In [ ]:
# TODO: para cada tarefa abaixo, teste temperaturas [0.0, 0.5, 1.0, 1.5]
# e escolha manualmente qual temperatura deu o melhor resultado.
# Justifique sua escolha.

tarefas = {
    "código":    "Escreva uma função Python que recebe uma lista e retorna a soma dos números pares.",
    "poesia":    "Escreva um haiku (3 linhas: 5-7-5 sílabas) sobre a chuva em Porto Alegre.",
    "factual":   "Quem foi Machado de Assis? Responda em 2 frases.",
}

for nome_tarefa, prompt_tarefa in tarefas.items():
    print(f"\n{'='*60}")
    print(f"TAREFA: {nome_tarefa.upper()}")
    print(f"{'='*60}")
    for temp in [0.0, 0.5, 1.0, 1.5]:
        print(f"\n[Temperatura {temp}]")
        print(perguntar(prompt_tarefa, temperatura=temp, max_tokens=150))

# Após rodar, preencha no respostas-03.md:
# Tarefa 'código'  → melhor temperatura: ___  Motivo: ___
# Tarefa 'poesia'  → melhor temperatura: ___  Motivo: ___
# Tarefa 'factual' → melhor temperatura: ___  Motivo: ___

---
## Respostas

Preencha o arquivo `respostas-03.md` com suas observacoes.

**Perguntas obrigatorias:**
1. O grafico foi linear (linha reta) ou teve alguma curva? O que isso sugere?
2. Para qual tipo de tarefa voce definitivamente usaria temperatura 0? E temperatura alta?
3. Existe uma temperatura "perfeita para tudo"? Justifique com exemplos do experimento.

**Pergunta desafio:**

4. Voce acha que alta criatividade = alta qualidade? Quando isso e verdade e quando nao e?